In [13]:
%pip install -q langchain-ollama

Note: you may need to restart the kernel to use updated packages.


In [14]:
!ollama list

]11;?\NAME          ID              SIZE      MODIFIED     
gemma4:e4b    c6eb396dbd59    9.6 GB    17 hours ago    


In [15]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model='gemma4:e4b')
llm.invoke("프랑스의 수도가 어디야?")

AIMessage(content='프랑스의 수도는 **파리**입니다. 😊', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-06-18T01:40:38.452326Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3552624625, 'load_duration': 231680250, 'prompt_eval_count': 24, 'prompt_eval_duration': 153069000, 'eval_count': 178, 'eval_duration': 3166462000, 'logprobs': None, 'model_name': 'gemma4:e4b'}, id='run--019ed863-1dd2-7192-8efa-1bb99d8b05e0-0', usage_metadata={'input_tokens': 24, 'output_tokens': 178, 'total_tokens': 202})

In [16]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_template = PromptTemplate(
    template="What is the capital of {country}? return the name of the city only",
    input_variables=["country"],
)

prompt = prompt_template.invoke({'country': 'Japan'})
result = llm.invoke(prompt)
print('raw result:', result)

output_parser = StrOutputParser()

parsed_result = output_parser.invoke(result)

print('parsed result:', parsed_result)

raw result: content='Tokyo' additional_kwargs={} response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-06-18T01:40:38.816952Z', 'done': True, 'done_reason': 'stop', 'total_duration': 329755042, 'load_duration': 234937042, 'prompt_eval_count': 30, 'prompt_eval_duration': 73882000, 'eval_count': 2, 'eval_duration': 19289000, 'logprobs': None, 'model_name': 'gemma4:e4b'} id='run--019ed863-2bd6-7e50-806f-e8bfd42fc0bb-0' usage_metadata={'input_tokens': 30, 'output_tokens': 2, 'total_tokens': 32}
parsed result: Tokyo


In [17]:
from langchain_core.output_parsers import JsonOutputParser

country_detail_prompt = PromptTemplate(
    template="""Give following information about {country}:
    - Capital
    - Population
    - Language
    - Currency

    return it in JSON format, and return the JSON dictionary only
    """,
    input_variables=["country"],
)

json_output_parser = JsonOutputParser()

json_output_parser.invoke(llm.invoke(country_detail_prompt.invoke({'country': 'Japan'})))


{'Capital': 'Tokyo',
 'Population': 'Approx. 125 million',
 'Language': 'Japanese',
 'Currency': 'Japanese Yen (JPY)'}

In [18]:
from pydantic import BaseModel, Field

class CountryDetail(BaseModel):
    capital: str = Field(description="The capital of the country")
    population: int = Field(description="The population of the country")
    language: str = Field(description="The language of the country")
    currency: str = Field(description="The currency of the country")

structured_llm = llm.with_structured_output(CountryDetail)

structured_llm.invoke(country_detail_prompt.invoke({'country': 'Japan'}))

CountryDetail(capital='Tokyo', population=125990378, language='Japanese', currency='Japanese Yen')

In [22]:
country_prompt = PromptTemplate(
    template="""Guess the name of the country based on the following information:
    {information}

    Return the name of the country only
    """,
    input_variables=["information"],
)

In [24]:
country_chain = country_prompt | llm | output_parser

country_chain.invoke({'information': '에펠탑으로 유명한 나라'})

'프랑스'

In [25]:
capital_chain = country_detail_prompt | llm | output_parser

final_chain = country_chain | capital_chain

In [27]:
final_chain.invoke({'information': '콜로세움으로 유명한 나라'})

'```json\n{\n  "country": "Italy",\n  "capital": "Rome",\n  "population": 59000000,\n  "language": "Italian",\n  "currency": "Euro"\n}\n```'

In [29]:
from langchain_core.runnables import RunnablePassthrough

real_final_chain = {'information': RunnablePassthrough()} | {'country': country_chain} | capital_chain

real_final_chain.invoke({'information':'김치의 나라'})

'```json\n{\n  "country": "대한민국",\n  "capital": "Seoul",\n  "population": "Approx. 51 million",\n  "language": "Korean",\n  "currency": "South Korean Won (KRW)"\n}\n```'